# Detección de Objetos con Faster R-CNN
Este notebook utiliza un modelo preentrenado de `torchvision` para detectar personas y objetos comunes en imágenes.

Funciona en local o en Google Colab, usando imágenes individuales o por lote desde una carpeta.

In [ ]:
# Instalación en Colab si es necesario
# !pip install torch torchvision matplotlib

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

## Cargar modelo preentrenado

In [ ]:
model = fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()
model.to(device)

## Lista de clases COCO (para mostrar etiquetas)

In [ ]:
COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag',
    'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite',
    'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana',
    'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table',
    'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock',
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

## Detección de objetos en lote desde una carpeta

In [ ]:
image_dir = "imagenes"  # carpeta con imágenes

for file in os.listdir(image_dir):
    if file.endswith(".jpg") or file.endswith(".png"):
        path = os.path.join(image_dir, file)
        img = Image.open(path).convert("RGB")
        img_tensor = F.to_tensor(img).to(device)
        with torch.no_grad():
            output = model([img_tensor])[0]

        fig, ax = plt.subplots(1, figsize=(12, 8))
        ax.imshow(img)
        ax.set_title(file)

        for box, label, score in zip(output['boxes'], output['labels'], output['scores']):
            if score > 0.6:
                box = box.to("cpu")
                xmin, ymin, xmax, ymax = box
                rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                         linewidth=2, edgecolor='lime', facecolor='none')
                ax.add_patch(rect)
                ax.text(xmin, ymin, f"{COCO_INSTANCE_CATEGORY_NAMES[label]}: {score:.2f}",
                        bbox=dict(facecolor='white', alpha=0.5))
        plt.axis('off')
        plt.show()